# Laboratorio 5

## Investigacion

## Capa RNN




### La **función de un RNN Elman (multi-layer)**

Para una **capa** de una red recurrente (Elman RNN) se aplica la siguiente regla:

$$
h_t = f\big(x_t W_{ih}^T + b_{ih} + h_{t-1} W_{hh}^T + b_{hh}\big)
$$

donde:

* $h_t$ → estado oculto en el tiempo $t$.
* $x_t$ → entrada en el tiempo $t$.
* $h_{t-1}$ → estado oculto en el tiempo anterior.
* $W_{ih}$ → pesos de entrada a oculto.
* $W_{hh}$ → pesos de recurrente (de oculto a oculto).
* $b_{ih}, b_{hh}$ → sesgos.
* $f$ → no linealidad, que puede ser `tanh` o `ReLU`.

Si se usa **varias capas** (multi-layer RNN), la salida $h_t^{(l)}$ de la capa $l$ se convierte en la entrada $x_t^{(l+1)}$ de la siguiente:

$$
h_t^{(l)} = f\big(h_t^{(l-1)} W_{ih}^{(l)T} + b_{ih}^{(l)} + h_{t-1}^{(l)} W_{hh}^{(l)T} + b_{hh}^{(l)}\big)
$$


Sabiendo eso podemos definir lo siguiente como los inputs y parametros que tiene nuestra RNN. 



### Input

* **x (la secuencia)**

  * El input principal es el tensor que contiene los datos de entrada.
  * Tiene siempre **3 dimensiones**:

    $$
    (L, N, H_{in})
    $$
  * **Si no hay batch (solo una secuencia):**

    $$
    (L, H_{in})
    $$
  * **Definiciones:**

    * $L$ = longitud de la secuencia (número de pasos de tiempo).
    * $N$ = tamaño del batch (número de secuencias procesadas en paralelo).
    * $H_{in}$ = dimensión de las features en cada paso (`input_size`).


* **hx (hidden state inicial)**

  * Este es el estado oculto inicial de la red.
  * Por defecto, si no se da, se inicializa en ceros.
  * Su forma depende de si la red es **unidireccional** o **bidireccional**, y cuántas capas tiene:

    * **Unbatched (una sola secuencia):**

      $$
      (D \cdot \text{num\_layers}, H_{out})
      $$
    * **Batched (varias secuencias):**

      $$
      (D \cdot \text{num\_layers}, N, H_{out})
      $$
  * **Definiciones:**

    * $D$ = número de direcciones (1 → unidireccional, 2 → bidireccional).
    * $\text{num\_layers}$ = número de capas RNN apiladas.
    * $N$ = batch size.
    * $H_{out}$ = dimensión del hidden state (`hidden_size`).




### Parametros

* **input\_size**

  * Número de características en la entrada de la red (dimensión de cada $x_t$).

* **hidden\_size**

  * Número de características en el estado oculto $h_t$.
  * Define cuánta “memoria” puede guardar la red.

* **num\_layers**

  * Número de capas recurrentes apiladas.
  * Ejemplo: si `num_layers=2`, la segunda capa recibe como entrada la salida de la primera.

* **nonlinearity**

  * Función no lineal usada para actualizar el estado oculto.
  * Puede ser `tanh` (por defecto) o `relu`.

* **bias**

  * Indica si se incluyen los términos de sesgo $b_{ih}$ y $b_{hh}$.
  * Si es `True`, la red agrega estos bias en los cálculos.

* **batch\_first**

  * Si es `True`, los tensores de entrada/salida usan la forma `(batch_size, seq_len, input_size)`.
  * Si es `False` (por defecto), se usa `(seq_len, batch_size, input_size)`.

* **dropout**

  * Si es distinto de 0, se aplica un *dropout* a las salidas de cada capa (excepto la última).
  * Controla el sobreajuste apagando aleatoriamente algunas conexiones.

* **bidirectional**

  * Si es `True`, la red se convierte en **RNN bidireccional**.
  * En ese caso, cada $h_t$ considera tanto el **pasado** como el **futuro**.
  * La dimensión de salida se duplica: `hidden_size * 2`.



### Output


* **output (la secuencia de salidas)**

  * Es un tensor que contiene las salidas $h_t$ de la **última capa de la RNN** para cada paso $t$.
  * Su forma depende de si hay batch y de si `batch_first` está activo:

    * **Unbatched (una sola secuencia):**

      $$
      (L, D \cdot H_{out})
      $$
    * **Batched, con `batch_first=False`:**

      $$
      (L, N, D \cdot H_{out})
      $$
    * **Batched, con `batch_first=True`:**

      $$
      (N, L, D \cdot H_{out})
      $$
  * **Definiciones:**

    * $L$ = longitud de la secuencia.
    * $N$ = tamaño del batch.
    * $D$ = número de direcciones (1 unidireccional, 2 bidireccional).
    * $H_{out}$ = dimensión del hidden state (`hidden_size`).
  * Si la entrada se dio como **PackedSequence** (`pack_padded_sequence`), la salida también será empaquetada.


* **h\_n (hidden state final)**

  * Es el **estado oculto final** de la red para cada capa y cada dirección.
  * Su forma depende de si hay batch:

    * **Unbatched (una sola secuencia):**

      $$
      (D \cdot \text{num\_layers}, H_{out})
      $$
    * **Batched (varias secuencias):**

      $$
      (D \cdot \text{num\_layers}, N, H_{out})
      $$
  * **Definiciones:**

    * $D$ = número de direcciones (1 → unidireccional, 2 → bidireccional).
    * $\text{num\_layers}$ = número de capas apiladas.
    * $N$ = batch size.
    * $H_{out}$ = dimensión del hidden state (`hidden_size`).



